# COMP-02 — Comparaison croisée avec les travaux externes

Objectif : croiser nos résultats (sirenisation + siretisation phases 1 à 3) avec un fichier de signalements produit sur un échantillon réduit, pour évaluer la convergence ou la divergence des verdicts.

**Fichier externe** : Excel multi-feuilles contenant des anomalies détectées sur des échantillons.
- Feuille `SignalementsurSIREN_PM` → sirenisation (clé : `numfinesspm`)
- Feuille `SignalementsurSIRET_EG` → siretisation (clé : `numfinessege`)

**Clés de jointure** :
- Sirenisation : `numfinesspm` (externe) ↔ `nmfinessej_stru` (nos résultats)
- Siretisation  : `numfinessege` (externe) ↔ `nmfinessetab_stru` (nos résultats)

**Verdicts de comparaison** :
- `CONVERGENT_REJET` : anomalie signalée par eux + REJETE de notre côté
- `CONVERGENT_DOUTEUX` : anomalie signalée + DOUTEUX de notre côté
- `DIVERGENT` : anomalie signalée **mais VALIDE de notre côté** → à investiguer
- `RÉSOLU_PAR_NOUS` : signalé comme non renseigné mais nous avons trouvé un candidat (P2/P3)
- `NON_TRAITÉ` : structure absente de nos résultats

## 1. Imports et configuration

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import re
import pandas as pd
from pathlib import Path

from src.comparaison import (
    charger_sirenisation_depuis_phases,
    charger_siretisation_depuis_phases,
)
from config.settings import (
    SN_PHASE1, SN_PHASE2, SN_PHASE3,
    ST_PHASE1, ST_PHASE2, ST_PHASE3,
    RESULTS_COMP_DIR,
)

# Chemin du fichier externe (à placer dans results/)
FICHIER_EXTERNE = RESULTS_COMP_DIR / 'FINESS_MiseenqualitéSIRENE_20250401.xlsx'

# Dossier de sortie pour l'export de la comparaison
RESULTS_COMP_DIR.mkdir(parents=True, exist_ok=True)
FICHIER_SORTIE = RESULTS_COMP_DIR / 'comparaison_croisee.xlsx'

VALIDES = {'VALIDE_FORT', 'VALIDE'}
PRIORITE_STATUT = {
    'VALIDE_FORT': 1, 'VALIDE': 2, 'DOUTEUX': 3, 'REJETE': 4,
    'SANS_CANDIDAT': 5, 'SANS_SIRET': 6, 'SANS_SIREN': 6,
    'SIRET_INCONNU': 7, 'SIREN_INCONNU': 7,
}

print('Configuration OK')
print(f'Fichier externe attendu : {FICHIER_EXTERNE}')

Configuration OK
Fichier externe attendu : /home/jovyan/work/projet_finess_sirene/results/comparaison/FINESS_MiseenqualitéSIRENE_20250401.xlsx


## 2. Chargement du fichier externe de signalements

In [2]:
# Chargement des deux feuilles du fichier externe
assert FICHIER_EXTERNE.exists(), f'Fichier introuvable : {FICHIER_EXTERNE}'

df_ext_sn = pd.read_excel(FICHIER_EXTERNE,
    sheet_name='SignalementsurSIREN_PM', dtype=str)
df_ext_st = pd.read_excel(FICHIER_EXTERNE,
    sheet_name='SignalementsurSIRET_EG', dtype=str)

print(f'Signalements sirenisation (PM)  : {len(df_ext_sn):,} lignes')
print(f'Colonnes : {list(df_ext_sn.columns)}')
print()
print(f'Signalements siretisation (EG) : {len(df_ext_st):,} lignes')
print(f'Colonnes : {list(df_ext_st.columns)}')

Signalements sirenisation (PM)  : 18,507 lignes
Colonnes : ['regionomlibellelong', 'departementom', 'departementomlibellelong', 'numfinesspm', 'denominationpm', 'siren', 'Description']

Signalements siretisation (EG) : 17,680 lignes
Colonnes : ['regionomlibellelong', 'departementom', 'departementomlibellelong', 'numfinessege', 'nomegecourt', 'siret', 'Description']


## 3. Typage automatique des anomalies

On extrait le type d'anomalie depuis la colonne `Description` pour pouvoir filtrer et regrouper ensuite.

In [ ]:
def typer_anomalie_sn(desc: str) -> str:
    """Classe le signalement sirenisation en 3 types."""
    if pd.isna(desc) or not desc:
        return 'INCONNU'
    d = str(desc).strip()
    if '999999999' in d:
        return 'SIREN_INVALIDE'        
    if 'non renseigné' in d.lower():
        return 'SIREN_NON_RENSEIGNE'   
    if 'fermée' in d.lower() or 'fermée' in d or 'fermé' in d.lower():
        return 'UL_FERMEE_SIRENE'    
    return 'AUTRE'

def typer_anomalie_st(desc: str) -> str:
    """Classe le signalement siretisation en types."""
    if pd.isna(desc) or not desc:
        return 'INCONNU'
    d = str(desc).strip()
    if 'fermé' in d.lower():
        return 'ETAB_FERME_SIRENE'    
    if 'non renseigné' in d.lower():
        return 'SIRET_NON_RENSEIGNE'
    if '999999999' in d:
        return 'SIRET_INVALIDE'
    return 'AUTRE'

# Normalisation de la clé de jointure (supprimer espaces, leading zeros, etc.)
def norm_id(v):
    if pd.isna(v) or str(v).strip() == '':
        return ''
    return re.sub(r'\s', '', str(v).strip()).lstrip('0').zfill(9)

# Appliquer le typage
df_ext_sn = df_ext_sn.copy()
df_ext_st = df_ext_st.copy()

df_ext_sn['type_anomalie'] = df_ext_sn['Description'].apply(typer_anomalie_sn)
df_ext_st['type_anomalie'] = df_ext_st['Description'].apply(typer_anomalie_st)

# Clé normalisée
df_ext_sn['cle_join'] = df_ext_sn['numfinesspm'].apply(norm_id)
df_ext_st['cle_join'] = df_ext_st['numfinessege'].apply(norm_id)

print('=== Répartition des types d\'anomalies sirenisation ===')
print(df_ext_sn['type_anomalie'].value_counts().to_string())
print()
print('=== Répartition des types d\'anomalies siretisation ===')
print(df_ext_st['type_anomalie'].value_counts().to_string())

=== Répartition des types d'anomalies sirenisation ===
type_anomalie
SIREN_NON_RENSEIGNE    15595
UL_FERMEE_SIRENE        1974
AUTRE                    831
SIREN_INVALIDE           107

=== Répartition des types d'anomalies siretisation ===
type_anomalie
ETAB_FERME_SIRENE    10705
AUTRE                 6973
SIRET_INVALIDE           2


## 4. Chargement de nos résultats (phases 1 à 3)

In [4]:
def statut_final_par_entite(df, id_col):
    """Pour chaque entité, retient le verdict de la dernière phase traitée."""
    VALIDES_LOCAL = {'VALIDE_FORT', 'VALIDE'}
    df = df.copy()
    df['phase'] = pd.to_numeric(df['phase'], errors='coerce')

    p1_v = df[(df['phase'] == 1) & (df['statut'].isin(VALIDES_LOCAL))]
    ids_v1 = set(p1_v[id_col].astype(str))

    p2_v = df[
        (df['phase'] == 2) &
        (df['statut'].isin(VALIDES_LOCAL)) &
        (~df[id_col].astype(str).isin(ids_v1))
    ]
    ids_resolus = ids_v1 | set(p2_v[id_col].astype(str))

    df_reste = df[~df[id_col].astype(str).isin(ids_resolus)]
    df_reste = (df_reste
        .sort_values('phase', ascending=False)
        .drop_duplicates(id_col, keep='first'))

    return pd.concat([p1_v, p2_v, df_reste], ignore_index=True)


def pick_premier_non_vide(row, cols):
    """Retourne la première valeur non vide parmi les colonnes."""
    for c in cols:
        v = row.get(c)
        if v is not None and pd.notna(v) and str(v).strip():
            return str(v).strip()
    return ''


# Charger les phases
df_sn_brut = charger_sirenisation_depuis_phases(SN_PHASE1, SN_PHASE2, SN_PHASE3)
df_st_brut = charger_siretisation_depuis_phases(ST_PHASE1, ST_PHASE2, ST_PHASE3)

# Verdict final par entité
df_sn = statut_final_par_entite(df_sn_brut, 'idstructure_stru')
df_st = statut_final_par_entite(df_st_brut, 'idstructure_stru')

# Reconstruire siren_retenu : P1 = siren_ul, P2 = siren_ref, P3 = siren_ref_app
df_sn['siren_retenu'] = df_sn.apply(
    lambda r: pick_premier_non_vide(r, ['siren_ul', 'siren_ref', 'siren_ref_app']),
    axis=1
)

# Reconstruire siret_retenu : P1 = siret_etab, P2 = siret_ref, P3 = siret_ref_app
df_st['siret_retenu'] = df_st.apply(
    lambda r: pick_premier_non_vide(r, ['siret_etab', 'siret_ref', 'siret_ref_app']),
    axis=1
)

# Clé normalisée côté nos résultats
df_sn['cle_join'] = df_sn['nmfinessej_stru'].apply(norm_id)
df_st['cle_join'] = df_st['nmfinessetab_stru'].apply(norm_id)

print(f'Nos résultats sirenisation : {len(df_sn):,} EJ (verdict final)')
print(df_sn['statut'].value_counts().to_string())
print()
print(f'Nos résultats siretisation : {len(df_st):,} EG (verdict final)')
print(df_st['statut'].value_counts().to_string())
print()

# Vérification rapide que siren_retenu est bien rempli
n_siren_rempli = (df_sn['siren_retenu'] != '').sum()
n_siret_rempli = (df_st['siret_retenu'] != '').sum()
print(f'siren_retenu renseigné : {n_siren_rempli:,} / {len(df_sn):,}')
print(f'siret_retenu renseigné : {n_siret_rempli:,} / {len(df_st):,}')

Nos résultats sirenisation : 54,097 EJ (verdict final)
statut
VALIDE         28234
VALIDE_FORT    23719
DOUTEUX         2098
REJETE            46

Nos résultats siretisation : 104,612 EG (verdict final)
statut
VALIDE         67191
VALIDE_FORT    26370
DOUTEUX        10070
REJETE           981

siren_retenu renseigné : 54,097 / 54,097
siret_retenu renseigné : 104,612 / 104,612


## 5. Logique de verdict croisé

In [5]:
def verdict_croise(statut_nos, type_anomalie_ext):
    """
    Classe la comparaison entre nos résultats et le signalement externe.

    Args:
        statut_nos      : notre statut (VALIDE_FORT, VALIDE, DOUTEUX, REJETE...)
        type_anomalie_ext : type d'anomalie du fichier externe

    Returns:
        str : verdict croisé
    """
    if pd.isna(statut_nos) or statut_nos == '':
        return 'NON_TRAITÉ'

    # Cas résolu : ils signalaient 'non renseigné' mais on a trouvé un candidat
    if type_anomalie_ext in ('SIREN_NON_RENSEIGNE', 'SIRET_NON_RENSEIGNE') \
            and statut_nos in VALIDES:
        return 'RÉSOLU_PAR_NOUS'

    # Divergence : ils signalent une anomalie mais on valide
    if statut_nos in VALIDES:
        return 'DIVERGENT'

    # Convergence rejet
    if statut_nos == 'REJETE':
        return 'CONVERGENT_REJET'

    # Convergence douteux
    if statut_nos == 'DOUTEUX':
        return 'CONVERGENT_DOUTEUX'

    # Pas de candidat / identifiant inconnu
    if statut_nos in ('SANS_SIREN', 'SIREN_INCONNU', 'SANS_SIRET',
                      'SIRET_INCONNU', 'SANS_CANDIDAT'):
        return 'CONVERGENT_REJET'

    return 'AUTRE'

## 6. Comparaison sirenisation

Jointure sur `numfinesspm` (externe) ↔ `nmfinessej_stru` (nos résultats).

In [6]:
# Colonnes utiles de nos résultats SN
cols_sn_nos = ['cle_join', 'nmfinessej_stru', 'raisonsociale_stru',
               'statut', 'phase', 'nmsiren_stru', 'siren_retenu']
cols_sn_nos = [c for c in cols_sn_nos if c in df_sn.columns]

# Jointure externe → nos résultats (left join pour garder toutes les anomalies)
df_comp_sn = df_ext_sn.merge(
    df_sn[cols_sn_nos].rename(columns={
        'statut':           'statut_nos',
        'phase':            'phase_nos',
        'siren_retenu':     'siren_retenu_nos',
        'raisonsociale_stru': 'nom_ej_nos',
    }),
    on='cle_join',
    how='left',
)

# Verdict croisé
df_comp_sn['verdict'] = df_comp_sn.apply(
    lambda r: verdict_croise(r.get('statut_nos'), r.get('type_anomalie')), axis=1
)

print(f'Lignes comparées : {len(df_comp_sn):,}')
print()
print('=== Répartition des verdicts (sirenisation) ===')
print(df_comp_sn['verdict'].value_counts().to_string())
print()
print('=== Verdicts par type d\'anomalie ===')
print(pd.crosstab(df_comp_sn['type_anomalie'], df_comp_sn['verdict']).to_string())

Lignes comparées : 18,507

=== Répartition des verdicts (sirenisation) ===
verdict
NON_TRAITÉ            14515
RÉSOLU_PAR_NOUS        1916
DIVERGENT              1535
CONVERGENT_DOUTEUX      526
CONVERGENT_REJET         15

=== Verdicts par type d'anomalie ===
verdict              CONVERGENT_DOUTEUX  CONVERGENT_REJET  DIVERGENT  NON_TRAITÉ  RÉSOLU_PAR_NOUS
type_anomalie                                                                                    
AUTRE                                14                 0         58         759                0
SIREN_INVALIDE                        0                 0          0         107                0
SIREN_NON_RENSEIGNE                 314                 2          0       13363             1916
UL_FERMEE_SIRENE                    198                13       1477         286                0


### 6.1. Focus sur les cas DIVERGENTS (sirenisation)

Ce sont les structures **signalées comme anomalie par l'externe** mais **validées par notre pipeline**. Ce sont les cas les plus importants à investiguer.

In [7]:
df_div_sn = df_comp_sn[df_comp_sn['verdict'] == 'DIVERGENT'].copy()
print(f'Cas DIVERGENTS sirenisation : {len(df_div_sn):,}')
print()
if len(df_div_sn) > 0:
    cols_aff = ['numfinesspm', 'denominationpm', 'type_anomalie',
                'siren', 'statut_nos', 'phase_nos', 'siren_retenu_nos', 'Description']
    cols_aff = [c for c in cols_aff if c in df_div_sn.columns]
    print(df_div_sn[cols_aff].to_string(index=False))
else:
    print('Aucun cas divergent — convergence totale sur la sirenisation.')

Cas DIVERGENTS sirenisation : 1,535

numfinesspm                         denominationpm    type_anomalie     siren  statut_nos  phase_nos siren_retenu_nos                                                                                                     Description
  010002731                      PHARMACIE GALENUS UL_FERMEE_SIRENE 347664351 VALIDE_FORT        2.0        888151719 PM_SMSSSE FINESSEJ = "010002731" ouverte mais unité légale SIREN = "347664351" correspondant fermée dans SIRENE
  010003713                    PHARMACIE DU LEVANT UL_FERMEE_SIRENE 502265754 VALIDE_FORT        2.0        899947998 PM_SMSSSE FINESSEJ = "010003713" ouverte mais unité légale SIREN = "502265754" correspondant fermée dans SIRENE
  010005197        PHARMACIE CENTRALE DE PREVESSIN UL_FERMEE_SIRENE 329972806 VALIDE_FORT        2.0        899839757 PM_SMSSSE FINESSEJ = "010005197" ouverte mais unité légale SIREN = "329972806" correspondant fermée dans SIRENE
  010005742                       PHARMACIE

### 6.2. Cas RÉSOLUS PAR NOUS (sirenisation)

Structures sans SIREN dans le fichier externe, pour lesquelles notre pipeline a trouvé un candidat en phase 2 ou 3.

In [8]:
df_res_sn = df_comp_sn[df_comp_sn['verdict'] == 'RÉSOLU_PAR_NOUS'].copy()
print(f'Cas résolus par nos travaux (sirenisation) : {len(df_res_sn):,}')
print()
if len(df_res_sn) > 0:
    cols_aff = ['numfinesspm', 'denominationpm', 'type_anomalie',
                'statut_nos', 'phase_nos', 'siren_retenu_nos']
    cols_aff = [c for c in cols_aff if c in df_res_sn.columns]
    print(df_res_sn[cols_aff].to_string(index=False))

Cas résolus par nos travaux (sirenisation) : 1,916

numfinesspm                         denominationpm       type_anomalie  statut_nos  phase_nos siren_retenu_nos
  010000628             ASS ASDOMI BOURG-EN-BRESSE SIREN_NON_RENSEIGNE      VALIDE        1.0        330674680
  010008688     ASSOCIATION DE GESTION DE LA MARPA SIREN_NON_RENSEIGNE VALIDE_FORT        2.0        753683093
  010008746     ASSOCIATION GESTION MARPA LE RENON SIREN_NON_RENSEIGNE VALIDE_FORT        2.0        529824278
  010008761  ASSOC GESTION MARPA CHALARONNN/CENTRE SIREN_NON_RENSEIGNE      VALIDE        1.0        751360157
  010008811 ASSOCIATION DE GESTION MARPA DE BRENOD SIREN_NON_RENSEIGNE      VALIDE        1.0        489606137
  010009546                MAISON DE SANTE BEYNOST SIREN_NON_RENSEIGNE      VALIDE        2.0        507587228
  010009900 CTRE DE FORMATION OPERATIONNELLE SANTE SIREN_NON_RENSEIGNE      VALIDE        2.0        349409912
  010010676  ASSOCIA DE GESTION MARPA LA REYSSOUZE SIREN_NON

## 7. Comparaison siretisation

Jointure sur `numfinessege` (externe) ↔ `nmfinessetab_stru` (nos résultats).

In [9]:
cols_st_nos = ['cle_join', 'nmfinessetab_stru', 'idstructure_stru',
               'raisonsociale_stru', 'statut', 'phase',
               'nmsiret_stru', 'siret_retenu']
cols_st_nos = [c for c in cols_st_nos if c in df_st.columns]

df_comp_st = df_ext_st.merge(
    df_st[cols_st_nos].rename(columns={
        'statut':           'statut_nos',
        'phase':            'phase_nos',
        'siret_retenu':     'siret_retenu_nos',
        'raisonsociale_stru': 'nom_eg_nos',
    }),
    on='cle_join',
    how='left',
)

df_comp_st['verdict'] = df_comp_st.apply(
    lambda r: verdict_croise(r.get('statut_nos'), r.get('type_anomalie')), axis=1
)

print(f'Lignes comparées : {len(df_comp_st):,}')
print()
print('=== Répartition des verdicts (siretisation) ===')
print(df_comp_st['verdict'].value_counts().to_string())
print()
print('=== Verdicts par type d\'anomalie ===')
print(pd.crosstab(df_comp_st['type_anomalie'], df_comp_st['verdict']).to_string())

Lignes comparées : 17,680

=== Répartition des verdicts (siretisation) ===
verdict
DIVERGENT             9743
NON_TRAITÉ            6305
CONVERGENT_DOUTEUX    1443
CONVERGENT_REJET       189

=== Verdicts par type d'anomalie ===
verdict            CONVERGENT_DOUTEUX  CONVERGENT_REJET  DIVERGENT  NON_TRAITÉ
type_anomalie                                                                 
AUTRE                             257                36       1759        4921
ETAB_FERME_SIRENE                1186               153       7984        1382
SIRET_INVALIDE                      0                 0          0           2


### 7.1. Focus sur les cas DIVERGENTS (siretisation)

In [10]:
df_div_st = df_comp_st[df_comp_st['verdict'] == 'DIVERGENT'].copy()
print(f'Cas DIVERGENTS siretisation : {len(df_div_st):,}')
print()
if len(df_div_st) > 0:
    cols_aff = ['numfinessege', 'nomegecourt', 'type_anomalie',
                'siret', 'statut_nos', 'phase_nos', 'siret_retenu_nos', 'Description']
    cols_aff = [c for c in cols_aff if c in df_div_st.columns]
    print(df_div_st[cols_aff].to_string(index=False))
else:
    print('Aucun cas divergent — convergence totale sur la siretisation.')

Cas DIVERGENTS siretisation : 9,743

numfinessege                            nomegecourt     type_anomalie          siret  statut_nos  phase_nos siret_retenu_nos                                                                                                               Description
   010002160        LBM NOVELAB MONTREVEL EN BRESSE             AUTRE 35172717700013      VALIDE        2.0   48992865500252 SIREN "351727177" de l'EGE FINESSET = "010002160" non égal au SIREN = "489928655" de sa PM_SMSSE  FINESSEJ = "690035159" 
   010002160        LBM NOVELAB MONTREVEL EN BRESSE ETAB_FERME_SIRENE 35172717700013      VALIDE        2.0   48992865500252            EGE FINESSET = "010002160" ouverte mais établissement SIRET = "35172717700013" correspondant fermé dans SIRENE
   010002244                            CATTP AGORA ETAB_FERME_SIRENE 77554456201452      VALIDE        2.0   22010001000234            EGE FINESSET = "010002244" ouverte mais établissement SIRET = "77554456201452" corresp

### 7.2. Cas RÉSOLUS PAR NOUS (siretisation)

Structures sans SIRET dans le fichier externe, pour lesquelles notre pipeline a trouvé un candidat en phase 2 ou 3.

In [11]:
df_res_st = df_comp_st[df_comp_st['verdict'] == 'RÉSOLU_PAR_NOUS'].copy()
print(f'Cas résolus par nos travaux (siretisation) : {len(df_res_st):,}')
print()
if len(df_res_st) > 0:
    cols_aff = ['numfinessege', 'nomegecourt', 'type_anomalie',
                'statut_nos', 'phase_nos', 'siret_retenu_nos']
    cols_aff = [c for c in cols_aff if c in df_res_st.columns]
    print(df_res_st[cols_aff].to_string(index=False))

Cas résolus par nos travaux (siretisation) : 0



## 8. Synthèse globale des deux comparaisons

In [12]:
def synthese_verdicts(df_comp, label):
    total = len(df_comp)
    vc = df_comp['verdict'].value_counts()
    lignes = []
    for v in ['DIVERGENT', 'CONVERGENT_REJET', 'CONVERGENT_DOUTEUX',
              'RÉSOLU_PAR_NOUS', 'NON_TRAITÉ', 'AUTRE']:
        n = vc.get(v, 0)
        pct = f'{n/total*100:.1f} %' if total > 0 else '—'
        lignes.append({'Volet': label, 'Verdict': v,
                       'Nombre': n, 'Part': pct})
    return pd.DataFrame(lignes)

df_synth = pd.concat([
    synthese_verdicts(df_comp_sn, 'Sirenisation (EJ)'),
    synthese_verdicts(df_comp_st, 'Siretisation (EG)'),
], ignore_index=True)

print('=== SYNTHÈSE GLOBALE DE LA COMPARAISON CROISÉE ===')
print(df_synth.to_string(index=False))
print()
# Points clés
n_div_sn = (df_comp_sn['verdict'] == 'DIVERGENT').sum()
n_div_st = (df_comp_st['verdict'] == 'DIVERGENT').sum()
n_res_sn = (df_comp_sn['verdict'] == 'RÉSOLU_PAR_NOUS').sum()
n_res_st = (df_comp_st['verdict'] == 'RÉSOLU_PAR_NOUS').sum()

print('─' * 60)
print('POINTS CLÉS :')
print(f'  Cas divergents à investiguer   : {n_div_sn + n_div_st:,} '
      f'({n_div_sn} SN + {n_div_st} ST)')
print(f'  Cas résolus par notre pipeline : {n_res_sn + n_res_st:,} '
      f'({n_res_sn} SN + {n_res_st} ST)')

=== SYNTHÈSE GLOBALE DE LA COMPARAISON CROISÉE ===
            Volet            Verdict  Nombre   Part
Sirenisation (EJ)          DIVERGENT    1535  8.3 %
Sirenisation (EJ)   CONVERGENT_REJET      15  0.1 %
Sirenisation (EJ) CONVERGENT_DOUTEUX     526  2.8 %
Sirenisation (EJ)    RÉSOLU_PAR_NOUS    1916 10.4 %
Sirenisation (EJ)         NON_TRAITÉ   14515 78.4 %
Sirenisation (EJ)              AUTRE       0  0.0 %
Siretisation (EG)          DIVERGENT    9743 55.1 %
Siretisation (EG)   CONVERGENT_REJET     189  1.1 %
Siretisation (EG) CONVERGENT_DOUTEUX    1443  8.2 %
Siretisation (EG)    RÉSOLU_PAR_NOUS       0  0.0 %
Siretisation (EG)         NON_TRAITÉ    6305 35.7 %
Siretisation (EG)              AUTRE       0  0.0 %

────────────────────────────────────────────────────────────
POINTS CLÉS :
  Cas divergents à investiguer   : 11,278 (1535 SN + 9743 ST)
  Cas résolus par notre pipeline : 1,916 (1916 SN + 0 ST)


## 9. Export Excel du résultat de comparaison

Le fichier produit contiendra 5 feuilles :
- `Synthese` : tableau des verdicts pour les deux volets
- `SN_Divergents` : cas sirenisation divergents (à investiguer)
- `SN_Resolus` : cas sirenisation résolus par notre pipeline
- `ST_Divergents` : cas siretisation divergents
- `ST_Resolus` : cas siretisation résolus

In [13]:
from openpyxl.styles import PatternFill, Font, Alignment

def style_entete(ws, couleur_hex):
    for cell in ws[1]:
        cell.fill  = PatternFill('solid', start_color=couleur_hex, end_color=couleur_hex)
        cell.font  = Font(bold=True, color='FFFFFF', name='Arial', size=10)
        cell.alignment = Alignment(horizontal='center', vertical='center')

with pd.ExcelWriter(FICHIER_SORTIE, engine='openpyxl') as writer:

    # Synthèse
    df_synth.to_excel(writer, sheet_name='Synthese', index=False)
    style_entete(writer.sheets['Synthese'], '1F3864')

    # Divergents sirenisation
    if len(df_div_sn) > 0:
        df_div_sn.to_excel(writer, sheet_name='SN_Divergents', index=False)
        style_entete(writer.sheets['SN_Divergents'], 'C00060')

    # Résolus sirenisation
    if len(df_res_sn) > 0:
        df_res_sn.to_excel(writer, sheet_name='SN_Resolus', index=False)
        style_entete(writer.sheets['SN_Resolus'], '1A7341')

    # Divergents siretisation
    if len(df_div_st) > 0:
        df_div_st.to_excel(writer, sheet_name='ST_Divergents', index=False)
        style_entete(writer.sheets['ST_Divergents'], 'C00060')

    # Résolus siretisation
    if len(df_res_st) > 0:
        df_res_st.to_excel(writer, sheet_name='ST_Resolus', index=False)
        style_entete(writer.sheets['ST_Resolus'], '1A7341')

    # Toutes les comparaisons (onglet complet)
    df_comp_sn.to_excel(writer, sheet_name='SN_Complet', index=False)
    style_entete(writer.sheets['SN_Complet'], '2E75B6')

    df_comp_st.to_excel(writer, sheet_name='ST_Complet', index=False)
    style_entete(writer.sheets['ST_Complet'], '2E75B6')

print(f'Export OK → {FICHIER_SORTIE}')

Export OK → /home/jovyan/work/projet_finess_sirene/results/comparaison/comparaison_croisee.xlsx
